#                                                                                                     Silver starts here

### First read each dataframe, i.e Population, Service, Status, Location and Demographics. Then we need to remove duplicates, normalize scores if needed. Standardize quarters. Deal with Null values in some columns. 

In [1]:
import argparse
from pyspark.sql import SparkSession, functions as F, Window
import os, re
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import col, split, trim, size, when, length, element_at, regexp_replace
import pandas
import numpy
import math


#### Read Demogrpahics from bronze delta lake

In [2]:
spark = (
    SparkSession.builder
    .appName("rename-and-save-1")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        # HDFS endpoint (from docker-compose: hdfs-namenode on port 9000)
        .config("spark.hadoop.fs.defaultFS", "hdfs://hdfs-namenode:9000")
        # small local-friendly shuffle
        .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [3]:


# ---- 1) paths ----
bronze_path = "hdfs:///data/delta/bronze/churn_demographics"  

# ---- 2) read bronze ----
df = spark.read.format("delta").load(bronze_path)



In [4]:
df.show()

+-----------+---------+------+---+--------+--------------+-------+----------+--------------+-----------+
|customer_id|count_val|gender|age|under_30|senior_citizen|married|dependents|num_dependents|ingest_date|
+-----------+---------+------+---+--------+--------------+-------+----------+--------------+-----------+
| 8779-QRDMV|        1|  Male| 78|   false|          true|  false|     false|             0| 2025-10-14|
| 7495-OOKFY|        1|Female| 74|   false|          true|   true|      true|             1| 2025-10-14|
| 1658-BYGOY|        1|  Male| 71|   false|          true|  false|      true|             3| 2025-10-14|
| 4598-XLKNJ|        1|Female| 78|   false|          true|   true|      true|             1| 2025-10-14|
| 4846-WHAFZ|        1|Female| 80|   false|          true|   true|      true|             1| 2025-10-14|
| 4412-YLTKF|        1|Female| 72|   false|          true|  false|      true|             1| 2025-10-14|
| 0390-DCFDQ|        1|Female| 76|   false|          tr

### Remove the count column as its redundant and convert the 

In [5]:
df = df.drop("count_val")


## The table already did look nice to begin with. it had no missing values, no duplicates or weird stuff. So we can add it to silver delta lake, we also need to make sure to store it as a table so that it can be queried

In [6]:
# TODO - add the dataframe as SQL table after consultation


silver_demographics_path = "hdfs:///data/delta/silver/churn_demographics"

# 1) Append the rows to the silver Delta path
(df
  .withColumn("loaded_at", F.current_timestamp())   
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")                
  .save(silver_demographics_path))



### Add the tables so they can be queried

In [ ]:
# 2) Register it in the catalog so you can SQL it (one-time)


silver_demographics_path = "hdfs:///data/delta/silver/churn_demographics"

spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS silver.churn_demographics
  USING DELTA
  LOCATION '{silver_demographics_path}'
""")



DataFrame[]

#### Test query

In [8]:
spark.sql("SELECT COUNT(*) AS rows FROM silver.churn_demographics").show()
spark.sql("""
  SELECT customer_id, gender, age, senior_citizen
  FROM silver.churn_demographics
  LIMIT 5
""").show(truncate=False)


+----+
|rows|
+----+
|7043|
+----+

+-----------+------+---+--------------+
|customer_id|gender|age|senior_citizen|
+-----------+------+---+--------------+
|8779-QRDMV |Male  |78 |true          |
|7495-OOKFY |Female|74 |true          |
|1658-BYGOY |Male  |71 |true          |
|4598-XLKNJ |Female|78 |true          |
|4846-WHAFZ |Female|80 |true          |
+-----------+------+---+--------------+



## Read location from bronze/churn_location

In [9]:


# ---- 1) paths ----
bronze_path = "hdfs:///data/delta/bronze/churn_location"  

# ---- 2) read bronze ----
df = spark.read.format("delta").load(bronze_path)



In [10]:
df.show()
df.printSchema()

+-----------+---------+-------------+----------+---------------+--------+---------+-----------+-----------+
|customer_id|count_val|      country|     state|           city|zip_code| latitude|  longitude|ingest_date|
+-----------+---------+-------------+----------+---------------+--------+---------+-----------+-----------+
| 8779-QRDMV|        1|United States|California|    Los Angeles|   90022| 34.02381|-118.156582| 2025-10-14|
| 7495-OOKFY|        1|United States|California|    Los Angeles|   90063|34.044271|-118.185237| 2025-10-14|
| 1658-BYGOY|        1|United States|California|    Los Angeles|   90065|34.108833|-118.229715| 2025-10-14|
| 4598-XLKNJ|        1|United States|California|      Inglewood|   90303|33.936291|-118.332639| 2025-10-14|
| 4846-WHAFZ|        1|United States|California|       Whittier|   90602|33.972119|-118.020188| 2025-10-14|
| 4412-YLTKF|        1|United States|California|    Pico Rivera|   90660|33.989524|-118.089299| 2025-10-14|
| 0390-DCFDQ|        1|Unite

## There is only one country in this dataframe namely United States so we can drop the country column. There is only one State namely California so we can drop this column.

In [11]:


distinct_countries=df.select('country').distinct()
distinct_states=df.select('state').distinct()

print("distinct countries")
distinct_countries.show()
print("distinct states ")
distinct_states.show()

distinct countries
+-------------+
|      country|
+-------------+
|United States|
+-------------+

distinct states 
+----------+
|     state|
+----------+
|California|
+----------+



In [12]:
df=df.drop('country').drop('state')
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- count_val: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- ingest_date: date (nullable = true)



### Check for duplicates 

In [13]:

column_to_check = "customer_id" 

duplicate_counts_df = df.groupBy(column_to_check).count()

duplicate_entries = duplicate_counts_df.filter(F.col("count") > 1)

print(f"Duplicate entries found in the '{column_to_check}' column:")
duplicate_entries.show(truncate=False)

total_duplicate_rows = duplicate_entries.select(F.sum("count")).collect()[0][0]
if total_duplicate_rows:
    total_duplicate_copies = total_duplicate_rows - duplicate_entries.count()
    print(f"\nTotal number of duplicate copies in this column: {total_duplicate_copies}")

Duplicate entries found in the 'customer_id' column:
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



## Everything looks nice we can add this to the silver/location and also make sure it queryable 

In [14]:


silver_location_path = "hdfs:///data/delta/silver/churn_location"

# 1) Append the rows to the silver Delta path
(df
  .withColumn("loaded_at", F.current_timestamp())   
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")                
  .save(silver_location_path))


In [15]:
# 2) Register it in the catalog so you can SQL it (one-time)
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS silver.churn_location
  USING DELTA
  LOCATION '{silver_location_path}'
""")


DataFrame[]

In [16]:
spark.sql("SELECT COUNT(*) AS rows FROM silver.churn_location").show()
spark.sql("""
  SELECT *
  FROM silver.churn_location
  LIMIT 5
""").show(truncate=False)


+----+
|rows|
+----+
|7043|
+----+

+-----------+---------+-----------+--------+---------+-----------+-----------+--------------------------+
|customer_id|count_val|city       |zip_code|latitude |longitude  |ingest_date|loaded_at                 |
+-----------+---------+-----------+--------+---------+-----------+-----------+--------------------------+
|8779-QRDMV |1        |Los Angeles|90022   |34.02381 |-118.156582|2025-10-14 |2025-10-15 09:46:02.083625|
|7495-OOKFY |1        |Los Angeles|90063   |34.044271|-118.185237|2025-10-14 |2025-10-15 09:46:02.083625|
|1658-BYGOY |1        |Los Angeles|90065   |34.108833|-118.229715|2025-10-14 |2025-10-15 09:46:02.083625|
|4598-XLKNJ |1        |Inglewood  |90303   |33.936291|-118.332639|2025-10-14 |2025-10-15 09:46:02.083625|
|4846-WHAFZ |1        |Whittier   |90602   |33.972119|-118.020188|2025-10-14 |2025-10-15 09:46:02.083625|
+-----------+---------+-----------+--------+---------+-----------+-----------+--------------------------+



# Now the population dataframe

#### Read it 

In [17]:
# ---- 1) paths ----
bronze_path = "hdfs:///data/delta/bronze/churn_population"  

# ---- 2) read bronze ----
df = spark.read.format("delta").load(bronze_path)

In [18]:
df.show()

+---+--------+----------+-----------+
| id|zip_code|population|ingest_date|
+---+--------+----------+-----------+
|  1|   90001|     54492| 2025-10-14|
|  2|   90002|     44586| 2025-10-14|
|  3|   90003|     58198| 2025-10-14|
|  4|   90004|     67852| 2025-10-14|
|  5|   90005|     43019| 2025-10-14|
|  6|   90006|     62784| 2025-10-14|
|  7|   90007|     45025| 2025-10-14|
|  8|   90008|     30852| 2025-10-14|
|  9|   90010|      1957| 2025-10-14|
| 10|   90011|    101215| 2025-10-14|
| 11|   90012|     30596| 2025-10-14|
| 12|   90013|      9732| 2025-10-14|
| 13|   90014|      3524| 2025-10-14|
| 14|   90015|     15140| 2025-10-14|
| 15|   90016|     46984| 2025-10-14|
| 16|   90017|     20692| 2025-10-14|
| 17|   90018|     47143| 2025-10-14|
| 18|   90019|     67520| 2025-10-14|
| 19|   90020|     42394| 2025-10-14|
| 20|   90021|      3012| 2025-10-14|
+---+--------+----------+-----------+
only showing top 20 rows



### Was already clean in the bronze level, the issue is with the id column, they dont match any of the other columns in our dataframes/tables? Should we drop the id-column ? 

#### Add it to the silver/churn_population anyways and make it queryable

In [19]:
silver_location_path = "hdfs:///data/delta/silver/churn_population"

# 1) Append the rows to the silver Delta path
(df
  .withColumn("loaded_at", F.current_timestamp())   
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")                
  .save(silver_location_path))

In [20]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS silver.churn_population
  USING DELTA
  LOCATION '{silver_location_path}'
""")

DataFrame[]

In [21]:
spark.sql("SELECT COUNT(*) AS rows FROM silver.churn_population").show()
spark.sql("""
  SELECT *
  FROM silver.churn_population
  LIMIT 5
""").show(truncate=False)

+----+
|rows|
+----+
|1671|
+----+

+---+--------+----------+-----------+-------------------------+
|id |zip_code|population|ingest_date|loaded_at                |
+---+--------+----------+-----------+-------------------------+
|1  |90001   |54492     |2025-10-14 |2025-10-15 09:46:12.92482|
|2  |90002   |44586     |2025-10-14 |2025-10-15 09:46:12.92482|
|3  |90003   |58198     |2025-10-14 |2025-10-15 09:46:12.92482|
|4  |90004   |67852     |2025-10-14 |2025-10-15 09:46:12.92482|
|5  |90005   |43019     |2025-10-14 |2025-10-15 09:46:12.92482|
+---+--------+----------+-----------+-------------------------+



In [22]:
spark.sql(""" SELECT * 
FROM silver.churn_population
WHERE population>40000 """).show()

+---+--------+----------+-----------+--------------------+
| id|zip_code|population|ingest_date|           loaded_at|
+---+--------+----------+-----------+--------------------+
|  1|   90001|     54492| 2025-10-14|2025-10-15 09:46:...|
|  2|   90002|     44586| 2025-10-14|2025-10-15 09:46:...|
|  3|   90003|     58198| 2025-10-14|2025-10-15 09:46:...|
|  4|   90004|     67852| 2025-10-14|2025-10-15 09:46:...|
|  5|   90005|     43019| 2025-10-14|2025-10-15 09:46:...|
|  6|   90006|     62784| 2025-10-14|2025-10-15 09:46:...|
|  7|   90007|     45025| 2025-10-14|2025-10-15 09:46:...|
| 10|   90011|    101215| 2025-10-14|2025-10-15 09:46:...|
| 15|   90016|     46984| 2025-10-14|2025-10-15 09:46:...|
| 17|   90018|     47143| 2025-10-14|2025-10-15 09:46:...|
| 18|   90019|     67520| 2025-10-14|2025-10-15 09:46:...|
| 19|   90020|     42394| 2025-10-14|2025-10-15 09:46:...|
| 21|   90022|     68701| 2025-10-14|2025-10-15 09:46:...|
| 22|   90023|     47487| 2025-10-14|2025-10-15 09:46:..

## Now let's do the services dataframe 

In [32]:
# ---- 1) paths ----
bronze_path = "hdfs:///data/delta/bronze/churn_services"  

# ---- 2) read bronze ----
df = spark.read.format("delta").load(bronze_path)

In [33]:
df.show(10)

+-----------+---------+-------+-----------------+-------------+----------------+-------+-------------+---------------------------------+--------------+----------------+-------------+-----------------------+---------------+-------------+----------------------+--------------------+------------+----------------+---------------+--------------+--------------+-----------------+---------------+--------------+-------------+-------------+------------------------+---------------------------+-------------+-----------+
|customer_id|count_val|quarter|referred_a_friend|num_referrals|tenure_in_months|  offer|phone_service|avg_monthly_long_distance_charges|multiple_lines|internet_service|internet_type|avg_monthly_gb_download|online_security|online_backup|device_protection_plan|premium_tech_support|streaming_tv|streaming_movies|streaming_music|unlimited_data|      contract|paperless_billing| payment_method|monthly_charge|total_charges|total_refunds|total_extra_data_charges|total_long_distance_charges|t

## How many missing values in the services dataframe ? There are 1526 missing values for internet_type and 3877 missing values for offer.

In [34]:

columns_to_check = df.columns

# 2. Build the list of aggregation expressions
missing_value_exprs = []
for c in columns_to_check:
    # Condition: Is the value genuinely NULL OR is the lowercase value equal to the string 'none'?
    condition = (F.col(c).isNull()) | (F.lower(F.col(c)) == 'none')
    
    # Sum the result of the condition (1 if true, 0 if false) and alias it
    missing_value_exprs.append(F.sum(F.when(condition, 1).otherwise(0)).alias(c))

# 3. Apply the aggregation to the DataFrame
missing_values_count = df.select(*missing_value_exprs)

# 4. Display the results
print("Count of ALL missing entries (NULL or 'None') per column:")
missing_values_count.select("internet_type","offer").show(truncate=False)

Count of ALL missing entries (NULL or 'None') per column:
+-------------+-----+
|internet_type|offer|
+-------------+-----+
|1526         |3877 |
+-------------+-----+



## Change the offer and internet_type features into categorical values and keep none as it is. 

In [43]:

# start from bronze services
bronze = df  # your cleaned bronze-level services DataFrame

# 1️⃣ Define the mapping dictionaries
offer_map = {
    "Offer A": 0,
    "Offer B": 1,
    "Offer C": 2,
    "Offer D": 3,
    "Offer E": 4
}

internet_map = {
    "DSL": 0,
    "Fiber Optic": 1,
    "Cable": 2
}


# simpler version: using CASE expression logic
df = (bronze
    .withColumn("offer_code",
        F.when(F.col("offer")=="Offer A",0)
         .when(F.col("offer")=="Offer B",1)
         .when(F.col("offer")=="Offer C",2)
         .when(F.col("offer")=="Offer D",3)
         .when(F.col("offer")=="Offer E",4)
         .otherwise(None)
    )
    .withColumn("internet_type_code",
        F.when(F.col("internet_type")=="DSL",0)
         .when(F.col("internet_type")=="Fiber Optic",1)
         .when(F.col("internet_type")=="Cable",2)
         .otherwise(None)
    )
)

# 3️⃣ Quick check
df.select("offer","offer_code","internet_type","internet_type_code").show()


+-------+----------+-------------+------------------+
|  offer|offer_code|internet_type|internet_type_code|
+-------+----------+-------------+------------------+
|   None|      NULL|          DSL|                 0|
|Offer E|         4|  Fiber Optic|                 1|
|Offer D|         3|  Fiber Optic|                 1|
|Offer C|         2|  Fiber Optic|                 1|
|Offer C|         2|  Fiber Optic|                 1|
|Offer C|         2|  Fiber Optic|                 1|
|Offer E|         4|  Fiber Optic|                 1|
|Offer B|         1|          DSL|                 0|
|Offer D|         3|  Fiber Optic|                 1|
|Offer E|         4|  Fiber Optic|                 1|
|Offer D|         3|  Fiber Optic|                 1|
|Offer D|         3|  Fiber Optic|                 1|
|Offer E|         4|          DSL|                 0|
|Offer E|         4|          DSL|                 0|
|Offer D|         3|        Cable|                 2|
|Offer D|         3|  Fiber 

# We write services dataframe to the silver data lake and make it queryable

In [44]:
silver_location_path = "hdfs:///data/delta/silver/churn_services"

# 1) Append the rows to the silver Delta path
(df
  .withColumn("loaded_at", F.current_timestamp())   
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")                
  .save(silver_location_path))

In [47]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS silver.services
  USING DELTA
  LOCATION '{silver_location_path}'
""")

DataFrame[]

In [52]:
spark.sql("SELECT COUNT(*) AS rows FROM silver.services").show()
spark.sql("""
  SELECT *
  FROM silver.services
  LIMIT 5
""").show(truncate=False)

+----+
|rows|
+----+
|7043|
+----+

+-----------+---------+-------+-----------------+-------------+----------------+-------+-------------+---------------------------------+--------------+----------------+-------------+-----------------------+---------------+-------------+----------------------+--------------------+------------+----------------+---------------+--------------+--------------+-----------------+---------------+--------------+-------------+-------------+------------------------+---------------------------+-------------+-----------+----------+------------------+--------------------------+
|customer_id|count_val|quarter|referred_a_friend|num_referrals|tenure_in_months|offer  |phone_service|avg_monthly_long_distance_charges|multiple_lines|internet_service|internet_type|avg_monthly_gb_download|online_security|online_backup|device_protection_plan|premium_tech_support|streaming_tv|streaming_movies|streaming_music|unlimited_data|contract      |paperless_billing|payment_method |mont

## Now we do status dataframe

In [53]:
# ---- 1) paths ----
bronze_path = "hdfs:///data/delta/bronze/churn_status"  

# ---- 2) read bronze ----
df = spark.read.format("delta").load(bronze_path)

In [55]:
df.show()

+-----------+---------+-------+------------------+---------------+-----------+-----------+-----------+----+---------------+--------------------+----------+-------------------+-----------+
|customer_id|count_val|quarter|satisfaction_score|customer_status|churn_label|churn_value|churn_score|cltv| churn_category|        churn_reason|churn_flag|customer_status_std|ingest_date|
+-----------+---------+-------+------------------+---------------+-----------+-----------+-----------+----+---------------+--------------------+----------+-------------------+-----------+
| 0002-ORFBO|        1|     Q3|                 3|         Stayed|         No|          0|         65|2205|           NULL|                NULL|     false|             stayed| 2025-10-14|
| 0003-MKNFE|        1|     Q3|                 5|         Stayed|         No|          0|         66|5414|           NULL|                NULL|     false|             stayed| 2025-10-14|
| 0004-TLHLJ|        1|     Q3|                 1|        Ch

In [58]:
df=df.drop('count_val')

In [59]:
df.show()

+-----------+-------+------------------+---------------+-----------+-----------+-----------+----+---------------+--------------------+----------+-------------------+-----------+
|customer_id|quarter|satisfaction_score|customer_status|churn_label|churn_value|churn_score|cltv| churn_category|        churn_reason|churn_flag|customer_status_std|ingest_date|
+-----------+-------+------------------+---------------+-----------+-----------+-----------+----+---------------+--------------------+----------+-------------------+-----------+
| 0002-ORFBO|     Q3|                 3|         Stayed|         No|          0|         65|2205|           NULL|                NULL|     false|             stayed| 2025-10-14|
| 0003-MKNFE|     Q3|                 5|         Stayed|         No|          0|         66|5414|           NULL|                NULL|     false|             stayed| 2025-10-14|
| 0004-TLHLJ|     Q3|                 1|        Churned|        Yes|          1|         71|4479|     Competit

In [61]:
df.select('customer_status','churn_label','churn_category','churn_reason','customer_status_std').show()

+---------------+-----------+---------------+--------------------+-------------------+
|customer_status|churn_label| churn_category|        churn_reason|customer_status_std|
+---------------+-----------+---------------+--------------------+-------------------+
|         Stayed|         No|           NULL|                NULL|             stayed|
|         Stayed|         No|           NULL|                NULL|             stayed|
|        Churned|        Yes|     Competitor|Competitor had be...|            churned|
|        Churned|        Yes|Dissatisfaction|Product dissatisf...|            churned|
|        Churned|        Yes|Dissatisfaction| Network reliability|            churned|
|         Stayed|         No|           NULL|                NULL|             stayed|
|         Stayed|         No|           NULL|                NULL|             stayed|
|         Stayed|         No|           NULL|                NULL|             stayed|
|         Stayed|         No|           NUL

In [64]:

columns_to_check = df.columns

# 2. Build the list of aggregation expressions
missing_value_exprs = []
for c in columns_to_check:
    # Condition: Is the value genuinely NULL OR is the lowercase value equal to the string 'none'?
    condition = (F.col(c).isNull()) | (F.lower(F.col(c)) == 'none')
    
    # Sum the result of the condition (1 if true, 0 if false) and alias it
    missing_value_exprs.append(F.sum(F.when(condition, 1).otherwise(0)).alias(c))

# 3. Apply the aggregation to the DataFrame
missing_values_count = df.select(*missing_value_exprs)

# 4. Display the results
print("Count of ALL missing entries (NULL or 'None') per column:")
missing_values_count.show(truncate=False)

Count of ALL missing entries (NULL or 'None') per column:
+-----------+-------+------------------+---------------+-----------+-----------+-----------+----+--------------+------------+----------+-------------------+-----------+
|customer_id|quarter|satisfaction_score|customer_status|churn_label|churn_value|churn_score|cltv|churn_category|churn_reason|churn_flag|customer_status_std|ingest_date|
+-----------+-------+------------------+---------------+-----------+-----------+-----------+----+--------------+------------+----------+-------------------+-----------+
|0          |0      |0                 |0              |0          |0          |0          |0   |5174          |5174        |0         |0                  |0          |
+-----------+-------+------------------+---------------+-----------+-----------+-----------+----+--------------+------------+----------+-------------------+-----------+



In [66]:

# joined=0, stayed=1, churned=2
status_order = {"joined": 0, "stayed": 1, "churned": 2}

# build a Spark map literal once
status_map = F.create_map(
    *[x for kv in status_order.items() for x in (F.lit(kv[0]), F.lit(kv[1]))]
)

status_silver = (
    df  
    .withColumn("customer_status_code", status_map[F.col("customer_status_std")].cast("int"))
)

#  sanity check
status_silver.show(10, truncate=False)



+-----------+-------+------------------+---------------+-----------+-----------+-----------+----+---------------+-----------------------------+----------+-------------------+-----------+--------------------+
|customer_id|quarter|satisfaction_score|customer_status|churn_label|churn_value|churn_score|cltv|churn_category |churn_reason                 |churn_flag|customer_status_std|ingest_date|customer_status_code|
+-----------+-------+------------------+---------------+-----------+-----------+-----------+----+---------------+-----------------------------+----------+-------------------+-----------+--------------------+
|0002-ORFBO |Q3     |3                 |Stayed         |No         |0          |65         |2205|NULL           |NULL                         |false     |stayed             |2025-10-14 |1                   |
|0003-MKNFE |Q3     |5                 |Stayed         |No         |0          |66         |5414|NULL           |NULL                         |false     |stayed        

# Now we write to silver delta 

In [67]:
silver_path = "hdfs:///data/delta/silver/churn_status"
(status_silver
  .write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema","true")
  .save(silver_path))



In [68]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql(f"""
  CREATE TABLE IF NOT EXISTS silver.churn_status
  USING DELTA
  LOCATION '{silver_path}'
""")


DataFrame[]

In [69]:
spark.sql("""
SELECT customer_status_std, customer_status_code, COUNT(*) AS n
FROM silver.churn_status
GROUP BY 1,2
ORDER BY customer_status_code
""").show()

spark.sql("""
SELECT churn_flag, AVG(churn_score) avg_score, AVG(cltv) avg_cltv
FROM silver.churn_status
GROUP BY churn_flag
""").show()

+-------------------+--------------------+----+
|customer_status_std|customer_status_code|   n|
+-------------------+--------------------+----+
|             joined|                   0| 454|
|             stayed|                   1|4720|
|            churned|                   2|1869|
+-------------------+--------------------+----+

+----------+-----------------+-----------------+
|churn_flag|        avg_score|         avg_cltv|
+----------+-----------------+-----------------+
|      true|81.77795612627074|4149.414660246121|
|     false|50.09818322381136|4490.921337456513|
+----------+-----------------+-----------------+

